# Capital budgeting: choosing among investments under a cash limit

Star Oil has five investments available. Each one has a net present value, and each one demands cash
now and again next year. There is 40 million available now and 20 million next year.

You cannot take all five — the cash runs out. **Which do you take?**

The question sounds like ranking. Whether ranking answers it is something you will test against two
models below: the investment with the highest NPV may be the one that eats the whole budget, and the
one with the best NPV *per dollar* may starve the others of next year's cash.

This notebook builds that model by hand, then asks a second question that changes the answer
substantially: **what if you cannot buy half an investment?**

## Setup: where the package lives

This notebook builds its models by hand and then checks them against `orteach`, the package in
`src/`. Run from a clone of the repository, `../../src` is right there. On Colab there is no clone
until this cell makes one, and no `gurobipy` until it installs it. Nothing here needs a secret.


In [1]:
import os, subprocess, sys

REPO_URL = "https://github.com/sear-labs/teaching-code"

try:
    import google.colab                      # noqa: F401 - succeeds only on Colab
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if ON_COLAB:
    if not os.path.isdir("/content/teaching-code"):
        subprocess.run(["git", "clone", "--quiet", REPO_URL, "/content/teaching-code"], check=True)
    os.chdir("/content/teaching-code/notebooks/11_decision_analysis")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "gurobipy>=11,<14"], check=True)

sys.path.insert(0, os.path.abspath(os.path.join("..", "..", "src")))
try:
    import orteach                            # noqa: F401
except ImportError:
    raise SystemExit("orteach not found: run this notebook from its own folder inside the repository, "
                     "so that ../../src exists.")
root = os.path.abspath(os.path.join("..", ".."))
print("package:", os.path.relpath(os.path.dirname(orteach.__file__), root))

package: src\orteach


## Licence setup

Nothing here needs a key: `pip install gurobipy` ships a size-limited licence and the models below
sit well inside it. A machine with its own licence file uses that instead, and on Colab three
secrets read from the key icon in the left sidebar are used when they are there — three named here,
none contained. The environment starts silent, so no licence number lands in an output cell.

In [2]:
import gurobipy as gp

env = gp.Env(empty=True)
env.setParam("OutputFlag", 0)        # start silent: the licence banner, and its licence number, stay out of the outputs
try:
    from google.colab import userdata
    try:
        env.setParam("WLSACCESSID", userdata.get("GRB_WLSACCESSID"))
        env.setParam("WLSSECRET",   userdata.get("GRB_WLSSECRET"))
        env.setParam("LICENSEID",   int(userdata.get("GRB_LICENSEID")))
        licence = "Colab Secrets (WLS)"
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        licence = "the size-limited licence pip ships"     # no key needed; see the note above
except ImportError:
    licence = "local gurobi.lic"
env.start()
print("licence:", licence)

licence: local gurobi.lic


## The investment table

Five investments, each with an NPV and two cash outflows. This is **instance data** — five rows
indexed by investment, with values the narration never names individually — so it lives in a file
that both this notebook and the package read.

That matters here more than usual. The version of this example these notebooks came from typed these
numbers **twice**: once into a dictionary and again, by hand, into the constraint expressions
(`11*x[1,0] + 53*x[2,0] + ...`). Editing the dictionary changed nothing about the model.

In [3]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join("..", "..", "src")))
from orteach.capital_budgeting import load_investments
from orteach import tolerance

investments = load_investments()

print(f"{'inv':>4} {'NPV':>8} {'cost now':>10} {'cost next':>10}")
for r in investments:
    print(f"{r['investment']:>4} {r['npv']:8.1f} {r['cost_t0']:10.1f} {r['cost_t1']:10.1f}")

 inv      NPV   cost now  cost next
   1     13.0       11.0        3.0
   2     16.0       53.0        6.0
   3     16.0        5.0        5.0
   4     14.0        5.0        1.0
   5     39.0       29.0       34.0


The two budgets are different in kind: they are **knobs**, single numbers the narration explains, so
they are written out here where you can see and change them.

In [4]:
budget_t0 = 40.0   # cash available now
budget_t1 = 20.0   # cash available next year

print(f"total cash across both periods : {budget_t0 + budget_t1:.0f}")
print(f"cost of buying all five        : "
      f"{sum(r['cost_t0'] + r['cost_t1'] for r in investments):.0f}")

total cash across both periods : 60
cost of buying all five        : 152


Buying everything would cost far more than is available, which is why this is a problem at all.

## Before you model: what does ranking suggest?

The instinct is to rank by NPV per dollar and buy down the list. Compute that ranking, buy down the list
until the cash runs out, and note the NPV you reach. You will compare it with both models.

**Predict:** which investment does this ranking put first?

In [5]:
print(f"{'inv':>4} {'NPV':>8} {'total cost':>11} {'NPV per $':>11}")
for r in sorted(investments, key=lambda r: -r["npv"] / (r["cost_t0"] + r["cost_t1"])):
    total = r["cost_t0"] + r["cost_t1"]
    print(f"{r['investment']:>4} {r['npv']:8.1f} {total:11.1f} {r['npv']/total:11.3f}")

 inv      NPV  total cost   NPV per $
   4     14.0         6.0       2.333
   3     16.0        10.0       1.600
   1     13.0        14.0       0.929
   5     39.0        63.0       0.619
   2     16.0        59.0       0.271


## The model, one piece at a time

One decision variable per investment: what **fraction** of it to buy. Star Oil can take a partial
stake, so these are continuous.

**The bound is the whole problem.** `ub=1.0` says you cannot buy an investment more than once. Drop
it and the model happily buys five copies of the best-value project and three of another, and reports
an NPV of more than double the truth — arithmetically correct and financially meaningless.

In [6]:
m = gp.Model(env=env)
tolerance.apply(m)          # solve as tightly as the assertion at the end claims
m.ModelSense = gp.GRB.MAXIMIZE

take = {}
for r in investments:
    take[r["investment"]] = m.addVar(lb=0.0, ub=1.0, obj=r["npv"],
                                     name=f"take[{r['investment']}]")
m.update()

print(f"{m.NumVars} variables, each bounded to [0, 1]")

5 variables, each bounded to [0, 1]


Now the two cash constraints. Each is built by summing over the table — so the numbers come from the
file, and there is exactly one copy of them.

In [7]:
c0 = m.addConstr(gp.quicksum(r["cost_t0"] * take[r["investment"]] for r in investments)
                 <= budget_t0, name="budget_t0")
c1 = m.addConstr(gp.quicksum(r["cost_t1"] * take[r["investment"]] for r in investments)
                 <= budget_t1, name="budget_t1")
m.update()

print(f"constraints: {m.NumConstrs}")
assert m.NumConstrs == 2, "expected exactly two budget constraints"
print("now:  ", m.getRow(c0))
print("next: ", m.getRow(c1))

constraints: 2
now:   11.0 take[1] + 53.0 take[2] + 5.0 take[3] + 5.0 take[4] + 29.0 take[5]
next:  3.0 take[1] + 6.0 take[2] + 5.0 take[3] + take[4] + 34.0 take[5]


**Predict before solving.** Your ranking above put one investment first. Will the optimal plan take
it in full? Will it take *any* investment in full? And will it spend all of both budgets?

In [8]:
m.optimize()

lp_npv = m.ObjVal
lp_take = {k: v.X for k, v in take.items()}

print(f"optimal NPV : {lp_npv:.3f}\n")
print(f"{'inv':>4} {'fraction':>10}")
for k, v in lp_take.items():
    print(f"{k:>4} {v:10.3f}")

optimal NPV : 57.449

 inv   fraction
   1      1.000
   2      0.201
   3      1.000
   4      1.000
   5      0.288


Three investments are taken whole and two are taken partially. Check what that did to the cash.

In [9]:
spend_0 = sum(r["cost_t0"] * lp_take[r["investment"]] for r in investments)
spend_1 = sum(r["cost_t1"] * lp_take[r["investment"]] for r in investments)

print(f"spent now  : {spend_0:6.2f} of {budget_t0:.0f}")
print(f"spent next : {spend_1:6.2f} of {budget_t1:.0f}")

spent now  :  40.00 of 40
spent next :  20.00 of 20


Both budgets are spent to the last dollar. Compare this plan with buying down your ranking: where
does the ranking's plan leave money unspent, and which stake did the model take partially in order to
use it?

## The second question: what if you cannot buy half a project?

Partial stakes are sometimes real and sometimes a modelling fiction. If each investment is a drilling
programme you either fund or do not, the fractions are meaningless and the answer above is not
executable.

One change: the variables become binary. Everything else is identical.

**Predict — this is the one worth writing down.** The fractional plan earned 57.4. Will the
all-or-nothing plan earn a little less, or a lot less? And will it still spend both budgets?

In [10]:
m2 = gp.Model(env=env)
tolerance.apply(m2)         # same tolerances as the package, or the check is meaningless
m2.ModelSense = gp.GRB.MAXIMIZE

take2 = {}
for r in investments:
    take2[r["investment"]] = m2.addVar(vtype=gp.GRB.BINARY, obj=r["npv"],
                                       name=f"take[{r['investment']}]")
m2.addConstr(gp.quicksum(r["cost_t0"] * take2[r["investment"]] for r in investments)
             <= budget_t0, name="budget_t0")
m2.addConstr(gp.quicksum(r["cost_t1"] * take2[r["investment"]] for r in investments)
             <= budget_t1, name="budget_t1")
m2.optimize()

ip_npv = m2.ObjVal
ip_take = {k: v.X for k, v in take2.items()}

print(f"optimal NPV : {ip_npv:.3f}\n")
print(f"{'inv':>4} {'take':>6}")
for k, v in ip_take.items():
    print(f"{k:>4} {'yes' if v > 0.5 else 'no':>6}")

optimal NPV : 43.000

 inv   take
   1    yes
   2     no
   3    yes
   4    yes
   5     no


Now look at the cash, which is the surprising part.

In [11]:
ip_spend_0 = sum(r["cost_t0"] * ip_take[r["investment"]] for r in investments)
ip_spend_1 = sum(r["cost_t1"] * ip_take[r["investment"]] for r in investments)

print(f"{'':12} {'now':>14} {'next':>14} {'NPV':>10}")
print(f"{'fractional':12} {spend_0:8.2f} / {budget_t0:<3.0f} {spend_1:8.2f} / {budget_t1:<3.0f} {lp_npv:10.3f}")
print(f"{'all or none':12} {ip_spend_0:8.2f} / {budget_t0:<3.0f} {ip_spend_1:8.2f} / {budget_t1:<3.0f} {ip_npv:10.3f}")
print()
print(f"cash left unspent now  : {budget_t0 - ip_spend_0:6.2f}")
print(f"cash left unspent next : {budget_t1 - ip_spend_1:6.2f}")

                        now           next        NPV
fractional      40.00 / 40     20.00 / 20      57.449
all or none     21.00 / 40      9.00 / 20      43.000

cash left unspent now  :  19.00
cash left unspent next :  11.00


The all-or-nothing plan **leaves exactly half of this year's money on the table** — not through
carelessness, but because no combination of whole investments fits the budgets any better. Look at
the two investments not taken: could either be bought with what is left? And how does this plan
compare with buying down your ranking in whole units?

The gap between the two answers is the **cost of indivisibility**: what it costs that investments come
in whole units.

In [12]:
gap = lp_npv - ip_npv
print(f"fractional NPV  : {lp_npv:8.3f}")
print(f"all-or-none NPV : {ip_npv:8.3f}")
print(f"cost of indivisibility : {gap:.3f}   ({100*gap/lp_npv:.1f}% of the LP value)")
assert ip_npv <= lp_npv + tolerance.FEASIBILITY_ATOL, "the integer answer cannot beat its own relaxation"
print("\nrelaxation bounds the integer answer, as it must")

fractional NPV  :   57.449
all-or-none NPV :   43.000
cost of indivisibility : 14.449   (25.2% of the LP value)

relaxation bounds the integer answer, as it must


That inequality is worth stating as a rule rather than an observation. The integer problem is the LP
**plus** extra restrictions, so its answer can never be better. The LP value is therefore always an
optimistic bound — useful for knowing how much you might still be leaving behind, and never
achievable unless it happens to come out whole on its own.

---

# Now the streamlined version

Both models differ by one argument — whether the variables are continuous or binary — so they belong
in one function with a flag, now that you have written each out separately.

`orteach.capital_budgeting` takes the investment table **as an argument** and never reads the file
itself, so editing a row above flows into both the hand-built model and the check below.

In [13]:
from orteach import capital_budgeting as cbg
from orteach.tolerance import AGREEMENT_RTOL, rel_diff

pkg_lp = cbg.solve_fractional(investments, budget_t0, budget_t1, env=env)
pkg_ip = cbg.solve_all_or_none(investments, budget_t0, budget_t1, env=env)

print(f"{pkg_lp.label:20} NPV {pkg_lp.npv:8.3f}")
print(f"{pkg_ip.label:20} NPV {pkg_ip.npv:8.3f}")
print(f"{'indivisibility':20}     {cbg.cost_of_indivisibility(pkg_lp, pkg_ip):8.3f}")

fractional (LP)      NPV   57.449
all or none (MIP)    NPV   43.000
indivisibility             14.449


## The agreement assertion

Same model, built twice on purpose. The check compares every number the notebook produced by hand
against the package's, including each investment fraction — not just the objective, because two
different plans can share an NPV.

One more thing the check relies on. Both sides solved at **tighter-than-default** solver
tolerances — `orteach.tolerance` sets Gurobi to `1e-9` optimality and feasibility, and a zero
MIP gap. Without that, a notebook asserting `1e-9` agreement against a solver that only promised
`1e-6` is not testing that the two models agree; it is testing that they took the same path on the
same machine, and it fails the first time it runs somewhere else. The tolerance itself is a single
constant in the package, imported here, not typed again.


In [14]:
checks = [("LP NPV", lp_npv, pkg_lp.npv), ("IP NPV", ip_npv, pkg_ip.npv)]
for r in investments:
    k = r["investment"]
    checks.append((f"LP take[{k}]", lp_take[k], pkg_lp.fractions[k]))
    checks.append((f"IP take[{k}]", ip_take[k], pkg_ip.fractions[k]))

worst = 0.0
for name, hand, packaged in checks:
    rel = rel_diff(hand, packaged)
    worst = max(worst, rel)
    print(f"{name:14} hand {hand:10.6f}   package {packaged:10.6f}   rel {rel:.2e}")

assert worst < AGREEMENT_RTOL, f"notebook and package disagree by {worst:.2e}"
print(f"\nnotebook and package agree to {worst:.1e}")

LP NPV         hand  57.449017   package  57.449017   rel 0.00e+00
IP NPV         hand  43.000000   package  43.000000   rel 0.00e+00
LP take[1]     hand   1.000000   package   1.000000   rel 0.00e+00
IP take[1]     hand   1.000000   package   1.000000   rel 0.00e+00
LP take[2]     hand   0.200860   package   0.200860   rel 0.00e+00
IP take[2]     hand   0.000000   package   0.000000   rel 0.00e+00
LP take[3]     hand   1.000000   package   1.000000   rel 0.00e+00
IP take[3]     hand   1.000000   package   1.000000   rel 0.00e+00
LP take[4]     hand   1.000000   package   1.000000   rel 0.00e+00
IP take[4]     hand   1.000000   package   1.000000   rel 0.00e+00
LP take[5]     hand   0.288084   package   0.288084   rel 0.00e+00
IP take[5]     hand   0.000000   package   0.000000   rel 0.00e+00

notebook and package agree to 0.0e+00


---

## Where to take this next

- Raise `budget_t1` from 20 to 30 and re-run. Does the all-or-nothing plan change, and does the cost
  of indivisibility grow or shrink?
- The unspent cash in the integer plan is real money. What would you have to add to the model to let
  it earn something — and does that change which projects get funded?
- The LP bound was 57.4 and the achievable answer 43.0. In a larger problem you cannot solve exactly,
  that gap is all you know about how good your answer is. What would make it tighter?